**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 28: DeepSeek R1 Case Study with GRPO

## 🎯 DeepSeek R1과 추론 모델

이 노트북에서는 DeepSeek R1의 학습 과정을 분석하고, **GRPO(Group Relative Policy Optimization)**를 사용하여  
Qwen2.5-1.5B 모델의 수학 추론 능력을 향상시키는 실습을 진행합니다.

### 학습 목표
- 🎯 DeepSeek R1의 학습 파이프라인 이해
- 1️⃣ GRPO 알고리즘의 상세 원리 파악
- 2️⃣ 보상 함수 설계 방법 학습
- 3️⃣ GRPO로 수학 추론 능력 향상 실습
- 4️⃣ Chain-of-Thought 추론 패턴 분석

### GPU 요구사항
- ✅ GPU 필수 (RTX 4060 8GB 이상)
- 모델: Qwen2.5-1.5B-Instruct (FP16)
- 예상 VRAM: ~6-8GB

---


In [1]:
# 필요한 라이브러리 임포트
import torch
import gc
import json
import re
import os
import random
import numpy as np
from pathlib import Path
from datetime import datetime

# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print("✅ 기본 라이브러리 임포트 완료")
print(f"📅 실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
else:
    print("❌ GPU를 사용할 수 없습니다. 이 노트북은 GPU가 필수입니다.")

✅ 기본 라이브러리 임포트 완료
📅 실행 시각: 2026-04-24 11:41:16
🔥 PyTorch: 2.10.0+cu128
🔥 CUDA: 12.8
✅ GPU: NVIDIA GeForce RTX 2070
[초기 상태] GPU: 0.0GB / 7.8GB


---
## 1️⃣ DeepSeek R1의 학습 과정 분석

### DeepSeek R1이란?

DeepSeek R1은 2024년 말 DeepSeek에서 발표한 **추론 특화 LLM**으로, OpenAI의 o1/o3에 필적하는 성능을 보여주었습니다.

### 핵심 혁신

```
🔑 핵심: 강화학습(GRPO)만으로 추론 능력을 획득!

기존 접근: SFT로 CoT 데이터 학습 → 추론 패턴 모방
DeepSeek R1: RL(GRPO)로 직접 추론 능력 발현 → 자발적 CoT 출현
```

### DeepSeek R1 학습 4단계

```
┌──────────────────────────────────────────────┐
│          DeepSeek R1 학습 파이프라인            │
├──────────────────────────────────────────────┤
│                                              │
│  Stage 1: Cold Start (소량 SFT)               │
│  └→ 기본적인 CoT 형식 학습                      │
│                                              │
│  Stage 2: 추론 중심 RL (GRPO)                  │
│  └→ 수학/코딩 문제로 추론 능력 강화              │
│  └→ 보상: 정답 여부 + 형식 보상                 │
│                                              │
│  Stage 3: Rejection Sampling + SFT            │
│  └→ RL 모델의 좋은 출력을 수집하여 재학습         │
│  └→ 일반 능력(글쓰기, 번역 등) 데이터 혼합       │
│                                              │
│  Stage 4: 전체 RL (GRPO)                      │
│  └→ 추론 + 일반 태스크 모두 보상 부여            │
│  └→ 안전성/유용성 보상 추가                     │
│                                              │
│  최종: 추론 + 일반 능력 모두 갖춘 모델            │
└──────────────────────────────────────────────┘
```

In [2]:
# DeepSeek R1 학습 단계 상세 분석
print("=" * 60)
print("📌 DeepSeek R1 학습 파이프라인 분석")
print("=" * 60)

stages = [
    {
        "name": "Stage 1: Cold Start SFT",
        "description": "소량의 CoT 데이터로 기본 형식 학습",
        "details": [
            "수천 개의 <think>...</think> 형식 예시 학습",
            "기본적인 사고 과정 표현 방법 습득",
            "목적: RL이 탐색할 수 있는 기본 능력 확보",
        ]
    },
    {
        "name": "Stage 2: 추론 중심 RL (GRPO)",
        "description": "수학/코딩 문제로 추론 능력 강화",
        "details": [
            "GRPO 알고리즘 사용 (Value Model 불필요)",
            "보상 1: 정답 여부 (정확한 답 = +1, 오답 = 0)",
            "보상 2: 형식 보상 (<think> 태그 사용 여부)",
            "놀라운 발견: 'aha moment' - 모델이 자발적으로 자기 검증 학습!",
        ]
    },
    {
        "name": "Stage 3: Rejection Sampling + SFT",
        "description": "좋은 출력을 수집하여 재학습",
        "details": [
            "Stage 2 모델에서 정답을 맞힌 출력 수집",
            "일반 태스크(글쓰기, 번역, QA) 데이터 혼합",
            "SFT로 추론+일반 능력 통합",
        ]
    },
    {
        "name": "Stage 4: 전체 RL (GRPO)",
        "description": "모든 태스크에 대해 RL 적용",
        "details": [
            "추론 태스크: 정답 기반 보상",
            "일반 태스크: LLM-as-judge 보상",
            "안전성/유용성 보상 추가",
        ]
    },
]

for stage in stages:
    print(f"\n🔷 {stage['name']}")
    print(f"   📋 {stage['description']}")
    for detail in stage['details']:
        print(f"      • {detail}")

📌 DeepSeek R1 학습 파이프라인 분석

🔷 Stage 1: Cold Start SFT
   📋 소량의 CoT 데이터로 기본 형식 학습
      • 수천 개의 <think>...</think> 형식 예시 학습
      • 기본적인 사고 과정 표현 방법 습득
      • 목적: RL이 탐색할 수 있는 기본 능력 확보

🔷 Stage 2: 추론 중심 RL (GRPO)
   📋 수학/코딩 문제로 추론 능력 강화
      • GRPO 알고리즘 사용 (Value Model 불필요)
      • 보상 1: 정답 여부 (정확한 답 = +1, 오답 = 0)
      • 보상 2: 형식 보상 (<think> 태그 사용 여부)
      • 놀라운 발견: 'aha moment' - 모델이 자발적으로 자기 검증 학습!

🔷 Stage 3: Rejection Sampling + SFT
   📋 좋은 출력을 수집하여 재학습
      • Stage 2 모델에서 정답을 맞힌 출력 수집
      • 일반 태스크(글쓰기, 번역, QA) 데이터 혼합
      • SFT로 추론+일반 능력 통합

🔷 Stage 4: 전체 RL (GRPO)
   📋 모든 태스크에 대해 RL 적용
      • 추론 태스크: 정답 기반 보상
      • 일반 태스크: LLM-as-judge 보상
      • 안전성/유용성 보상 추가


In [3]:
# DeepSeek R1의 "Aha Moment" 시연
print("=" * 60)
print("📌 DeepSeek R1의 'Aha Moment' 예시")
print("=" * 60)

print("""
🧠 "Aha Moment"란?
   RL 학습 중 모델이 자발적으로 자기 검증(self-verification) 능력을 획득하는 순간

📝 예시 - 수학 문제 풀이 과정:

<think>
문제: 27 × 14 = ?

풀이 시작:
27 × 14 = 27 × (10 + 4)
        = 27 × 10 + 27 × 4  
        = 270 + 108
        = 378

잠깐, 검증해보자.            ← 🎯 Aha Moment!
27 × 14 = (30 - 3) × 14    ← 다른 방법으로 검증
        = 420 - 42
        = 378 ✓              ← 일치 확인
</think>

27 × 14 = 378

💡 핵심: 이 자기 검증 능력은 SFT로 학습시킨 것이 아니라,
   RL(GRPO)를 통해 자연스럽게 발현된 것입니다!
   
   → 정답을 맞히면 보상을 받으므로, 검증 과정이 정답률을 높이는 데
     도움이 된다는 것을 모델이 스스로 학습한 것입니다.
""")

📌 DeepSeek R1의 'Aha Moment' 예시

🧠 "Aha Moment"란?
   RL 학습 중 모델이 자발적으로 자기 검증(self-verification) 능력을 획득하는 순간

📝 예시 - 수학 문제 풀이 과정:

<think>
문제: 27 × 14 = ?

풀이 시작:
27 × 14 = 27 × (10 + 4)
        = 27 × 10 + 27 × 4  
        = 270 + 108
        = 378

잠깐, 검증해보자.            ← 🎯 Aha Moment!
27 × 14 = (30 - 3) × 14    ← 다른 방법으로 검증
        = 420 - 42
        = 378 ✓              ← 일치 확인
</think>

27 × 14 = 378

💡 핵심: 이 자기 검증 능력은 SFT로 학습시킨 것이 아니라,
   RL(GRPO)를 통해 자연스럽게 발현된 것입니다!

   → 정답을 맞히면 보상을 받으므로, 검증 과정이 정답률을 높이는 데
     도움이 된다는 것을 모델이 스스로 학습한 것입니다.



---
## 2️⃣ GRPO 알고리즘 상세 (Group Relative Policy Optimization)

### GRPO 수학적 정의

```
GRPO 목적 함수:

J(θ) = E_q~P(Q) [ 1/G × Σ_{i=1}^{G} (
    min(
        r_i(θ) × Â_i,
        clip(r_i(θ), 1-ε, 1+ε) × Â_i
    ) - β × KL(π_θ || π_ref)
)]

여기서:
- q: 입력 프롬프트
- G: 그룹 크기 (각 프롬프트당 생성 개수)
- r_i(θ) = π_θ(o_i|q) / π_old(o_i|q): 정책 비율
- Â_i = (r_i - mean(r)) / std(r): 정규화된 이점
- ε: 클리핑 범위
- β: KL 페널티 강도
```

### PPO와의 핵심 차이

```
PPO:  Advantage = Value Function으로 계산 → Value Model 필요
GRPO: Advantage = 그룹 내 상대 비교로 계산 → Value Model 불필요!
```

### GRPO 알고리즘 의사코드

```python
for each batch of prompts Q:
    for each prompt q in Q:
        # 1. G개 응답 생성
        outputs = [generate(q) for _ in range(G)]
        
        # 2. 보상 계산
        rewards = [reward_fn(q, o) for o in outputs]
        
        # 3. 그룹 내 정규화
        advantages = (rewards - mean(rewards)) / std(rewards)
        
    # 4. PPO 스타일 업데이트 (클리핑 + KL 페널티)
    update_policy(outputs, advantages)
```

In [4]:
# GRPO 알고리즘 시뮬레이션
print("=" * 60)
print("📌 GRPO 알고리즘 시뮬레이션")
print("=" * 60)

def simulate_grpo_step(prompt, group_size=4):
    """
    GRPO 한 스텝을 시뮬레이션합니다.
    """
    print(f"\n📝 프롬프트: {prompt['question']}")
    print(f"   정답: {prompt['answer']}")
    print(f"   그룹 크기: {group_size}")
    
    # 1. G개 응답 생성 (시뮬레이션)
    print(f"\n   Step 1: {group_size}개 응답 생성")
    outputs = []
    for i in range(group_size):
        # 일부는 정답, 일부는 오답으로 시뮬레이션
        correct = random.random() > 0.4  # 60% 정답률
        if correct:
            answer = prompt['answer']
        else:
            answer = str(int(prompt['answer']) + random.choice([-2, -1, 1, 2]))
        outputs.append({"answer": answer, "correct": str(answer) == str(prompt['answer'])})
        status = "✅" if outputs[-1]["correct"] else "❌"
        print(f"      응답 {i+1}: {answer} {status}")
    
    # 2. 보상 계산
    print(f"\n   Step 2: 보상 계산")
    rewards = []
    for o in outputs:
        r = 1.0 if o["correct"] else 0.0
        rewards.append(r)
        print(f"      r = {r:.1f}")
    
    rewards = np.array(rewards)
    
    # 3. 그룹 내 정규화
    print(f"\n   Step 3: 그룹 내 정규화")
    mean_r = np.mean(rewards)
    std_r = np.std(rewards) + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    print(f"      평균: {mean_r:.3f}, 표준편차: {std_r:.3f}")
    for i, (adv, r) in enumerate(zip(advantages, rewards)):
        direction = "↑ 강화" if adv > 0 else "↓ 약화" if adv < 0 else "→ 유지"
        print(f"      Â_{i+1} = ({r:.1f} - {mean_r:.3f}) / {std_r:.3f} = {adv:+.3f} ({direction})")
    
    # 4. 정책 업데이트 방향
    print(f"\n   Step 4: 정책 업데이트 방향")
    for i, (o, adv) in enumerate(zip(outputs, advantages)):
        if adv > 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 증가 ↑ (정답)")
        elif adv < 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 감소 ↓ (오답)")
    
    return advantages

# 시뮬레이션 실행
random.seed(42)
prompts = [
    {"question": "15 + 27 = ?", "answer": "42"},
    {"question": "8 × 7 = ?", "answer": "56"},
]

for p in prompts:
    print("\n" + "=" * 50)
    simulate_grpo_step(p, group_size=4)

📌 GRPO 알고리즘 시뮬레이션


📝 프롬프트: 15 + 27 = ?
   정답: 42
   그룹 크기: 4

   Step 1: 4개 응답 생성
      응답 1: 42 ✅
      응답 2: 43 ❌
      응답 3: 41 ❌
      응답 4: 42 ✅

   Step 2: 보상 계산
      r = 1.0
      r = 0.0
      r = 0.0
      r = 1.0

   Step 3: 그룹 내 정규화
      평균: 0.500, 표준편차: 0.500
      Â_1 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_2 = (0.0 - 0.500) / 0.500 = -1.000 (↓ 약화)
      Â_3 = (0.0 - 0.500) / 0.500 = -1.000 (↓ 약화)
      Â_4 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)

   Step 4: 정책 업데이트 방향
      응답 1 (42): 확률 증가 ↑ (정답)
      응답 2 (43): 확률 감소 ↓ (오답)
      응답 3 (41): 확률 감소 ↓ (오답)
      응답 4 (42): 확률 증가 ↑ (정답)


📝 프롬프트: 8 × 7 = ?
   정답: 56
   그룹 크기: 4

   Step 1: 4개 응답 생성
      응답 1: 56 ✅
      응답 2: 56 ✅
      응답 3: 58 ❌
      응답 4: 54 ❌

   Step 2: 보상 계산
      r = 1.0
      r = 1.0
      r = 0.0
      r = 0.0

   Step 3: 그룹 내 정규화
      평균: 0.500, 표준편차: 0.500
      Â_1 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_2 = (1.0 - 0.500) / 0.500 = +1.000 (↑ 강화)
      Â_3 = (0.0 - 0.500) /

---
## 3️⃣ 환경 설정 및 Qwen2.5-1.5B 로드

In [5]:
import sys
sys.modules["vllm"] = None  # vllm 버전 호환 문제 방지

# 추가 라이브러리 임포트
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

print("✅ 모든 라이브러리 임포트 완료")
print("   • transformers, peft, trl (GRPOTrainer), datasets")

/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 모든 라이브러리 임포트 완료
   • transformers, peft, trl (GRPOTrainer), datasets


In [6]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("./outputs/grpo_training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("📌 GRPO 학습 설정")
print("=" * 60)
print(f"   모델: {MODEL_NAME}")
print(f"   정밀도: FP16 (양자화 없음)")
print(f"   출력: {OUTPUT_DIR}")

📌 GRPO 학습 설정
   모델: Qwen/Qwen2.5-1.5B-Instruct
   정밀도: FP16 (양자화 없음)
   출력: outputs/grpo_training


In [7]:
# 모델 및 토크나이저 로드
print("=" * 60)
print("📌 모델 로딩")
print("=" * 60)

print("🔄 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"   ✅ 토크나이저 로드 완료")

print("🔄 모델 로딩... (FP16)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"   ✅ 모델 로드 완료")
print_gpu_memory("모델 로드 후")

📌 모델 로딩
🔄 토크나이저 로딩...
   ✅ 토크나이저 로드 완료
🔄 모델 로딩... (FP16)


`torch_dtype` is deprecated! Use `dtype` instead!


   ✅ 모델 로드 완료
[모델 로드 후] GPU: 2.9GB / 7.8GB


In [8]:
# LoRA 설정
print("=" * 60)
print("📌 LoRA 설정")
print("=" * 60)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"   LoRA rank: {lora_config.r}")
print(f"   LoRA alpha: {lora_config.lora_alpha}")
print(f"   Target: {lora_config.target_modules}")

📌 LoRA 설정
   LoRA rank: 16
   LoRA alpha: 32
   Target: {'v_proj', 'o_proj', 'q_proj', 'k_proj'}


---
## 4️⃣ 수학 문제 데이터 준비 (간단한 산술/논리 문제)

GRPO 학습에 사용할 수학 문제 데이터를 준비합니다.  
검증 가능한 정답이 있는 문제를 사용합니다.

In [9]:
# 수학/추론 문제 데이터 생성
print("=" * 60)
print("📌 수학/추론 문제 데이터 생성")
print("=" * 60)

random.seed(42)

def generate_problems(n=100):
    """덧셈 + 문장제(덧셈) + 논리 추론(대소 비교) 문제 생성"""
    problems = []
    names = ["A", "B", "C", "D", "E"]
    
    for _ in range(n):
        problem_type = random.choice(["addition", "word_problem", "comparison"])
        
        if problem_type == "addition":
            a = random.randint(10, 99)
            b = random.randint(10, 99)
            question = f"{a} + {b}의 값은 얼마인가요?"
            answer = str(a + b)
            
        elif problem_type == "word_problem":
            templates = [
                ("사과가 {a}개 있었는데 {b}개를 더 샀습니다. 총 사과는 몇 개인가요?", lambda a, b: a + b),
                ("철수가 {a}원, 영희가 {b}원을 가지고 있습니다. 두 사람의 합계는?", lambda a, b: a + b),
                ("오전에 {a}명, 오후에 {b}명이 방문했습니다. 총 방문자 수는?", lambda a, b: a + b),
                ("첫 번째 상자에 {a}개, 두 번째 상자에 {b}개가 있습니다. 전체 개수는?", lambda a, b: a + b),
            ]
            template, calc = random.choice(templates)
            a = random.randint(5, 50)
            b = random.randint(5, 50)
            question = template.format(a=a, b=b)
            answer = str(calc(a, b))
            
        else:  # comparison (논리 추론)
            # A > B, B > C → A와 C의 관계는?
            vals = sorted(random.sample(range(1, 100), 3), reverse=True)
            n1, n2, n3 = random.sample(names[:3], 3)
            
            pattern = random.choice(["transitive", "reverse", "middle"])
            
            if pattern == "transitive":
                # A > B, B > C → A ? C
                question = f"{n1}은 {n2}보다 크고, {n2}은 {n3}보다 큽니다. {n1}과 {n3} 중 누가 더 큰가요?"
                answer = n1
            elif pattern == "reverse":
                # A < B, B < C → A ? C
                question = f"{n1}은 {n2}보다 작고, {n2}은 {n3}보다 작습니다. {n1}과 {n3} 중 누가 더 큰가요?"
                answer = n3
            else:
                # A > B, A > C → B ? C → 알 수 없음
                question = f"{n1}은 {n2}보다 크고, {n1}은 {n3}보다 큽니다. {n2}과 {n3} 중 누가 더 큰가요?"
                answer = "알 수 없음"
        
        prompt = (
            f"다음 문제를 단계별로 풀어주세요. "
            f"먼저 풀이 과정을 보여주고, 마지막에 '정답: [답]' 형식으로 답을 제시하세요.\n\n"
            f"문제: {question}"
        )
        
        problems.append({
            "prompt": prompt,
            "answer": answer,
            "type": problem_type,
        })
    
    return problems

# 문제 생성 또는 로드
MATH_DATA_PATH = str(OUTPUT_DIR / "math_problems.json")

if os.path.exists(MATH_DATA_PATH):
    print(f"💾 기존 문제 로드: {MATH_DATA_PATH}")
    with open(MATH_DATA_PATH, "r", encoding="utf-8") as f:
        math_problems = json.load(f)
else:
    math_problems = generate_problems(100)
    os.makedirs(str(OUTPUT_DIR), exist_ok=True)
    with open(MATH_DATA_PATH, "w", encoding="utf-8") as f:
        json.dump(math_problems, f, ensure_ascii=False, indent=2)
    print(f"💾 문제 저장: {MATH_DATA_PATH}")

print(f"\n✅ {len(math_problems)}개 문제 준비 완료")
print(f"\n📊 유형별 분포:")
type_counts = {}
for p in math_problems:
    type_counts[p['type']] = type_counts.get(p['type'], 0) + 1
for t, c in type_counts.items():
    print(f"   {t}: {c}개")

print(f"\n📋 예시 문제들:")
for p in math_problems[:6]:
    print(f"   [{p['type']}] {p['prompt'][60:120]}... → 정답: {p['answer']}")

📌 수학/추론 문제 데이터 생성
💾 문제 저장: outputs/grpo_training/math_problems.json

✅ 100개 문제 준비 완료

📊 유형별 분포:
   comparison: 32개
   addition: 39개
   word_problem: 29개

📋 예시 문제들:
   [comparison] 하세요.

문제: B은 A보다 크고, A은 C보다 큽니다. B과 C 중 누가 더 큰가요?... → 정답: B
   [comparison] 하세요.

문제: C은 A보다 크고, A은 B보다 큽니다. C과 B 중 누가 더 큰가요?... → 정답: C
   [addition] 하세요.

문제: 21 + 37의 값은 얼마인가요?... → 정답: 58
   [addition] 하세요.

문제: 74 + 87의 값은 얼마인가요?... → 정답: 161
   [addition] 하세요.

문제: 81 + 35의 값은 얼마인가요?... → 정답: 116
   [comparison] 하세요.

문제: B은 A보다 크고, B은 C보다 큽니다. A과 C 중 누가 더 큰가요?... → 정답: 알 수 없음


In [10]:
# GRPO 데이터셋 준비
print("=" * 60)
print("📌 GRPO 데이터셋 준비")
print("=" * 60)

SYSTEM_PROMPT = (
    "당신은 수학 문제를 단계별로 풀어주는 AI입니다. "
    "반드시 <think> 태그 안에 풀이 과정을 작성하고, "
    "태그를 닫은 후 '정답: [숫자]' 형식으로 최종 답을 제시하세요.\n\n"
    "예시:\n"
    "<think>\n"
    "15 + 27을 계산합니다.\n"
    "15 + 27 = 42\n"
    "</think>\n"
    "정답: 42"
)

grpo_data = []
for p in math_problems:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": p["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    grpo_data.append({
        "prompt": prompt_text,
        "answer": p["answer"],
    })

grpo_dataset = Dataset.from_list(grpo_data)

print(f"✅ GRPO 데이터셋 생성 완료: {len(grpo_dataset)}개 샘플")
print(f"\n📋 시스템 프롬프트 (DeepSeek R1 스타일):")
print(f"   <think> 태그로 추론 과정 → 정답 출력")
print(f"\n📋 첫 번째 프롬프트 미리보기:")
print(grpo_dataset[0]["prompt"][:300] + "...")

📌 GRPO 데이터셋 준비
✅ GRPO 데이터셋 생성 완료: 100개 샘플

📋 시스템 프롬프트 (DeepSeek R1 스타일):
   <think> 태그로 추론 과정 → 정답 출력

📋 첫 번째 프롬프트 미리보기:
<|im_start|>system
당신은 수학 문제를 단계별로 풀어주는 AI입니다. 반드시 <think> 태그 안에 풀이 과정을 작성하고, 태그를 닫은 후 '정답: [숫자]' 형식으로 최종 답을 제시하세요.

예시:
<think>
15 + 27을 계산합니다.
15 + 27 = 42
</think>
정답: 42<|im_end|>
<|im_start|>user
다음 문제를 단계별로 풀어주세요. 먼저 풀이 과정을 보여주고, 마지막에 '정답: [답]' 형식으로 답을 제시하세요.

문제: B은 A보다 크고, A은 C보다 큽니다. B과 C 중...


---
## 5️⃣ 보상 함수 정의 (정답 검증)

GRPO의 핵심은 **보상 함수(Reward Function)**입니다.  
DeepSeek R1처럼 규칙 기반 보상을 정의하되, **정답 정확도**에 가장 큰 가중치를 둡니다.

### 보상 설계

| 보상 | 조건 | 점수 |
|------|------|------|
| 정답 보상 | 최종 답이 정답과 일치 | **+3.0** (86%) |
| 형식 보상 | '정답: X' 형식 준수 | +0.2 |
| `<think>` 태그 | `<think>...</think>` 사용 | +0.2 |
| 풀이 보상 | `<think>` 안 풀이 30자+ | +0.1 |
| 오답 | 정답 불일치 | 0.0 |

**최대 3.5점**. 정답에 86%의 비중을 두어 "형식만 맞추고 답은 틀리는" 상황을 방지합니다.

In [11]:
# 보상 함수 정의
print("=" * 60)
print("📌 보상 함수 정의")
print("=" * 60)

def extract_answer(text: str) -> str:
    """모델 출력에서 정답을 추출 (숫자/이름/텍스트)"""
    # 패턴 1: '정답: X' (숫자, A/B/C, 알 수 없음 등)
    match = re.search(r'정답\s*[:：]\s*\[?([^\]\n]+?)\]?\s*$', text, re.MULTILINE)
    if match:
        return match.group(1).strip()
    # 패턴 2: '답: X'
    match = re.search(r'답\s*[:：]\s*\[?([^\]\n]+?)\]?\s*$', text, re.MULTILINE)
    if match:
        return match.group(1).strip()
    # 패턴 3: 마지막 줄의 A/B/C 또는 "알 수 없음"
    match = re.search(r'\b([A-C])\b|알\s*수\s*없음', text[-100:])
    if match:
        return match.group(0).replace(" ", "")
    # 패턴 4: 마지막 숫자
    numbers = re.findall(r'\b(\d+)\b', text)
    if numbers:
        return numbers[-1]
    return ""


def reward_function(completions, answer, **kwargs):
    """
    GRPO 보상 함수 (DeepSeek R1 스타일)
    - 정답: +3.0 (가장 중요)
    - 형식('정답:'): +0.2
    - <think>...</think> 사용: +0.2
    - <think> 안 풀이 과정 (30자+): +0.1
    최대 3.5점, 정답 비중 86%
    """
    rewards = []
    for completion, correct_answer in zip(completions, answer):
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)

        reward = 0.0

        # 1. 정답 보상 (+3.0) — 가장 중요
        extracted = extract_answer(text)
        correct_str = str(correct_answer).replace(" ", "")
        extracted_clean = extracted.replace(" ", "")
        if extracted_clean == correct_str:
            reward += 3.0

        # 2. 형식 보상 (+0.2)
        if re.search(r'정답\s*[:：]', text):
            reward += 0.2

        # 3. <think> 태그 보상 (+0.2)
        has_think_open = "<think>" in text
        has_think_close = "</think>" in text
        if has_think_open and has_think_close:
            reward += 0.2
        elif has_think_open:
            reward += 0.1

        # 4. 풀이 보상 (+0.1): <think> 안 30자+
        think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
        if think_match and len(think_match.group(1).strip()) > 30:
            reward += 0.1

        rewards.append(reward)
    return rewards


print("✅ 보상 함수 정의 완료")
print("\n📋 보상 체계 (정답 중심으로 재설계):")
print("   • 정답: +3.0 (86%)")
print("   • 형식 준수 ('정답: X'): +0.2")
print("   • <think>...</think> 태그: +0.2")
print("   • <think> 안 풀이 30자+: +0.1")
print("   • 최대 보상: 3.5")

# 보상 함수 테스트 (덧셈 + 비교 문제 모두)
print("\n📊 보상 함수 테스트:")
test_completions = [
    "<think>\n15+27=42\n</think>\n정답: 42",                                    # 덧셈 정답+형식
    "15+27=42\n정답: 42",                                                        # 덧셈 정답, think 없음
    "<think>\nA<C, C<B, 따라서 A<B\n</think>\n정답: B",                         # 비교 정답
    "<think>\nB가 크다고 판단됨\n</think>\n정답: A",                             # 비교 오답
    "<think>\n알 수 없다\n</think>\n정답: 알 수 없음",                           # 비교 "알 수 없음"
]
test_answers = ["42", "42", "B", "B", "알 수 없음"]

test_rewards = reward_function(test_completions, test_answers)
for comp, ans, rew in zip(test_completions, test_answers, test_rewards):
    print(f"   답={ans}, 응답: {comp[:50]}... → 보상: {rew:.2f}")

📌 보상 함수 정의
✅ 보상 함수 정의 완료

📋 보상 체계 (정답 중심으로 재설계):
   • 정답: +3.0 (86%)
   • 형식 준수 ('정답: X'): +0.2
   • <think>...</think> 태그: +0.2
   • <think> 안 풀이 30자+: +0.1
   • 최대 보상: 3.5

📊 보상 함수 테스트:
   답=42, 응답: <think>
15+27=42
</think>
정답: 42... → 보상: 3.40
   답=42, 응답: 15+27=42
정답: 42... → 보상: 3.20
   답=B, 응답: <think>
A<C, C<B, 따라서 A<B
</think>
정답: B... → 보상: 3.40
   답=B, 응답: <think>
B가 크다고 판단됨
</think>
정답: A... → 보상: 0.40
   답=알 수 없음, 응답: <think>
알 수 없다
</think>
정답: 알 수 없음... → 보상: 3.40


---
## 6️⃣ GRPO 학습 실행 (GRPOTrainer from trl)

TRL의 GRPOTrainer를 사용하여 GRPO 학습을 실행합니다.

In [12]:
# 학습 전 모델 성능 측정 (베이스라인)
print("=" * 60)
print("📌 학습 전 베이스라인 성능 측정")
print("=" * 60)

def evaluate_math_model(model, tokenizer, problems, n_eval=10):
    """수학 문제에 대한 모델 성능 평가"""
    model.eval()
    
    # 추론을 위해 모든 파라미터를 bfloat16으로 통일
    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.float16)
    
    correct = 0
    total = min(n_eval, len(problems))
    results = []
    
    for i in range(total):
        prompt = problems[i]["prompt"]
        correct_answer = problems[i]["answer"]
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        extracted = extract_answer(response)
        is_correct = extracted == str(correct_answer)
        
        if is_correct:
            correct += 1
        
        results.append({
            "correct_answer": correct_answer,
            "extracted": extracted,
            "is_correct": is_correct,
            "response": response[:200],
        })
    
    accuracy = correct / total * 100
    return accuracy, results

# 베이스라인 평가
print("🔄 베이스라인 성능 측정 중...")
baseline_acc, baseline_results = evaluate_math_model(model, tokenizer, grpo_data, n_eval=10)

print(f"\n📊 베이스라인 정확도: {baseline_acc:.1f}%")
print(f"\n📋 상세 결과:")
for i, r in enumerate(baseline_results, 1):
    status = "✅" if r["is_correct"] else "❌"
    print(f"   {i}. 정답: {r['correct_answer']}, 추출: {r['extracted']}, {status}")
    print(f"      응답: {r['response'][:100]}...")

print_gpu_memory("베이스라인 평가 후")


📌 학습 전 베이스라인 성능 측정
🔄 베이스라인 성능 측정 중...

📊 베이스라인 정확도: 80.0%

📋 상세 결과:
   1. 정답: B, 추출: B, ✅
      응답: <think>
B은 A보다 크므로, 즉 A < B.
A는 C보다 큽니다. 따라서, A > C.
따라서, B과 C 중에서 B가 더 큽니다.
</think>

정답: B...
   2. 정답: C, 추출: C, ✅
      응답: <think>
C는 A보다 크므로, 따라서 C > A.
A는 B보다 큽니다.
따라서 C와 B의 관계를 비교할 때는 우선 C가 가장 중요하다.
따라서 C > B.
</think>

...
   3. 정답: 58, 추출: 58, ✅
      응답: <think>
먼저 두 숫자를 세로로 정렬하겠습니다.
21
+ 37
------
</think>

두 수를 더해보겠습니다.
21 + 37 = 58

</think>

정답: 58...
   4. 정답: 161, 추출: 161, ✅
      응답: <think>
먼저 두 수 74와 87를 더해보겠습니다.
</think>

<think>
74 + 87 = 161
</think>

정답: 161...
   5. 정답: 116, 추출: 96, ❌
      응답: <think>
먼저 두 숫자를 더해봅니다.
81 + 35
</think>

81과 35를 더하면 다음과 같습니다:

81
+ 35
-----
96

따라서, 81 + 35의 결과는...
   6. 정답: 알 수 없음, 추출: A, ❌
      응답: <think>
A와 C 사이의 관계를 이해하기 위해 다음과 같이 사칙연산을 통해 비교해보겠습니다:

1) A와 C 간의 관계를 확인하기 위해 A - C를 계산합니다.
   A - ...
   7. 정답: 64, 추출: 64, ✅
      응답: <think>
사과의 초기 수량은 15개이고 추가로 샀던 개수는 49개입니다. 이 두 숫자를 더하면 총 사과의 수를 알 수 있습니다.
</think>

<answer>
15

In [13]:
# GRPO 학습 설정 (RTX 4060 최적화)
print("=" * 60)
print("📌 GRPO 학습 설정")
print("=" * 60)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR / "grpo_checkpoint"),
    
    # 학습 설정
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-6,
    
    # GRPO 특화 설정
    num_generations=4,           # 그룹 크기 (G) - 메모리 제한으로 4
    max_completion_length=256,   # 최대 생성 길이
    max_prompt_length=512,       # 최대 프롬프트 길이
    
    # 메모리 최적화
    fp16=True,
    optim="paged_adamw_8bit",
    
    # 로깅
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    
    # 기타
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
)

print("📋 GRPO 핵심 설정:")
print(f"   num_generations (G): {grpo_config.num_generations}")
print(f"   max_completion_length: {grpo_config.max_completion_length}")
print(f"   batch_size: {grpo_config.per_device_train_batch_size}")
print(f"   gradient_accumulation: {grpo_config.gradient_accumulation_steps}")
print(f"   learning_rate: {grpo_config.learning_rate}")
print(f"   epochs: {grpo_config.num_train_epochs}")

print("\n💡 RTX 4060 메모리 제한으로:")
print("   • num_generations=4 (DeepSeek은 64 사용)")
print("   • max_completion_length=256")
print("   • per_device_train_batch_size=1")


📌 GRPO 학습 설정
📋 GRPO 핵심 설정:
   num_generations (G): 4
   max_completion_length: 256
   batch_size: 1
   gradient_accumulation: 8
   learning_rate: 1e-06
   epochs: 1

💡 RTX 4060 메모리 제한으로:
   • num_generations=4 (DeepSeek은 64 사용)
   • max_completion_length=256
   • per_device_train_batch_size=1


In [14]:
# GRPO Trainer 생성 및 학습
print("=" * 60)
print("📌 GRPO 학습 시작")
print("=" * 60)
print_gpu_memory("학습 시작 전")

# GRPOTrainer 생성
grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=reward_function,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("\n🔥 GRPO 학습 시작...")
print("   (이 과정은 RTX 4060에서 약 7-10분 소요됩니다)")

grpo_result = grpo_trainer.train()

print(f"\n✅ GRPO 학습 완료!")
print(f"   Training Loss: {grpo_result.training_loss:.4f}")
print(f"   Steps: {grpo_result.global_step}")
print_gpu_memory("학습 완료 후")


📌 GRPO 학습 시작
[학습 시작 전] GPU: 2.9GB / 7.8GB


/usr/bin/ld: cannot find -lcufile: 그런 파일이나 디렉터리가 없습니다
collect2: error: ld returned 1 exit status
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



🔥 GRPO 학습 시작...
   (이 과정은 RTX 4060에서 약 7-10분 소요됩니다)


Step,Training Loss
1,0.120600
2,0.060400
3,0.231100
4,0.337500
5,-0.130700
6,0.101500
7,-0.029200
8,0.009900
9,0.048900
10,0.097200



✅ GRPO 학습 완료!
   Training Loss: 0.0787
   Steps: 50
[학습 완료 후] GPU: 2.9GB / 7.8GB


In [15]:
# GRPO 모델 저장
print("🔄 GRPO 모델 저장 중...")
grpo_trainer.save_model(str(OUTPUT_DIR / "grpo_checkpoint"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "grpo_checkpoint"))
print(f"✅ GRPO 모델 저장 완료: {OUTPUT_DIR / 'grpo_checkpoint'}")

🔄 GRPO 모델 저장 중...
✅ GRPO 모델 저장 완료: outputs/grpo_training/grpo_checkpoint


---
## 7️⃣ 학습 전후 추론 능력 비교

In [16]:
# 학습 후 성능 측정
print("=" * 60)
print("📌 학습 전후 추론 능력 비교")
print("=" * 60)

print("🔄 학습 후 성능 측정 중...")
grpo_trainer.model.eval()

# 새로운 테스트 문제
test_problems_new = []
random.seed(999)
for _ in range(10):
    a = random.randint(10, 50)
    b = random.randint(10, 50)
    answer = str(a + b)
    op = "+"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 문제를 단계별로 풀어주세요.\n\n문제: {a} + {b}의 값은 얼마인가요?"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    test_problems_new.append({"prompt": prompt_text, "answer": answer})

after_acc, after_results = evaluate_math_model(
    grpo_trainer.model, tokenizer, test_problems_new, n_eval=10
)

print(f"\n📊 성능 비교:")
print(f"   학습 전 (베이스라인): {baseline_acc:.1f}%")
print(f"   학습 후 (GRPO):      {after_acc:.1f}%")
improvement = after_acc - baseline_acc
print(f"   향상:                {improvement:+.1f}%")

print(f"\n📋 학습 후 상세 결과:")
for i, r in enumerate(after_results, 1):
    status = "✅" if r["is_correct"] else "❌"
    print(f"   {i}. 정답: {r['correct_answer']}, 추출: {r['extracted']}, {status}")
    print(f"      응답: {r['response'][:150]}...")

# <think> 태그 사용 비율
think_count = sum(1 for r in after_results if "<think>" in r["response"])
print(f"\n📊 <think> 태그 사용: {think_count}/{len(after_results)}개 응답")


📌 학습 전후 추론 능력 비교
🔄 학습 후 성능 측정 중...

📊 성능 비교:
   학습 전 (베이스라인): 80.0%
   학습 후 (GRPO):      100.0%
   향상:                +20.0%

📋 학습 후 상세 결과:
   1. 정답: 61, 추출: 61, ✅
      응답: <think>
먼저, 두 숫자를 각각 더합니다.
15 + 46 = 61
</think>
정답: 61...
   2. 정답: 90, 추출: 90, ✅
      응답: <think>
먼저 두 숫자를 비교하여 더하기 전에 더할 수 있는 조건이 있다면 이를 먼저 확인합니다.
46과 44 중에서 44는 46보다 작으므로 더해야 하는 수입니다.
</think>

<think>
더해야 하는 수 44와 46을 더합니다.
44 + 46 = 90
...
   3. 정답: 81, 추출: 81, ✅
      응답: <think>
먼저 두 숫자를 각각 분석해 보겠습니다.
41과 40을 더하기 전에, 먼저 41을 더할 때는 41에 40을 더하는 것이 맞습니다.
즉, 41 + 40 = 81
</think>

정답: 81...
   4. 정답: 48, 추출: 48, ✅
      응답: <think>
먼저 두 숫자를 더해보겠습니다.
18 + 30 = 
</think>
정답: 48...
   5. 정답: 38, 추출: 38, ✅
      응답: <think>
먼저 두 숫자 16과 22를 더해보겠습니다.
16 + 22 = 38
</think>
정답: 38...
   6. 정답: 45, 추출: 45, ✅
      응답: <think>
먼저 두 숫자를 더해보겠습니다.
19 + 26 = 
</think>

수학적으로 계산하면 다음과 같습니다:

19 + 26 = 45

정답: 45...
   7. 정답: 59, 추출: 59, ✅
      응답: <think>
먼저 두 숫자를 각각 분해합니다.
46은 40과 6을 합칩니다.
13은 10과 3을 합칩니다.
</think>

<think>

---
## 8️⃣ Chain-of-Thought 추론 패턴 분석

GRPO 학습 후 모델이 어떤 추론 패턴을 보이는지 분석합니다.

In [17]:
# Chain-of-Thought 패턴 분석
print("=" * 60)
print("📌 Chain-of-Thought 추론 패턴 분석 (<think> 태그)")
print("=" * 60)

analysis_problems = [
    "37 + 48의 값은 얼마인가요?",
    "A은 B보다 크고, B은 C보다 큽니다. A과 C 중 누가 더 큰가요?",
    "사과가 15개 있었는데 8개를 더 샀습니다. 총 사과는 몇 개인가요?",
]

for prompt_text in analysis_problems:
    print(f"\n{'='*50}")
    print(f"❓ 문제: {prompt_text}")
    print(f"{'='*50}")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 문제를 단계별로 풀어주세요.\n\n문제: {prompt_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(grpo_trainer.model.device)
    with torch.no_grad():
        outputs = grpo_trainer.model.generate(
            **inputs, max_new_tokens=256,
            temperature=0.3, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    print(f"\n🤖 모델 응답:")
    print(f"   {response}")

    # <think> 태그 패턴 분석
    print(f"\n📊 패턴 분석:")
    has_think = "<think>" in response and "</think>" in response
    think_match = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
    has_answer_format = bool(re.search(r'정답\s*[:：]', response))

    print(f"   <think> 태그 사용: {'✅' if has_think else '❌'}")
    if think_match:
        think_content = think_match.group(1).strip()
        print(f"   <think> 내용 길이: {len(think_content)}자")
        print(f"   <think> 내용: {think_content[:150]}")
    print(f"   정답 형식: {'✅' if has_answer_format else '❌'}")
    print(f"   전체 응답 길이: {len(response)}자")


📌 Chain-of-Thought 추론 패턴 분석 (<think> 태그)

❓ 문제: 37 + 48의 값은 얼마인가요?

🤖 모델 응답:
   <think>
먼저 두 숫자를 각각 더해보겠습니다.
37 + 48 = 85
</think>
정답: 85<|im_end|>

📊 패턴 분석:
   <think> 태그 사용: ✅
   <think> 내용 길이: 33자
   <think> 내용: 먼저 두 숫자를 각각 더해보겠습니다.
37 + 48 = 85
   정답 형식: ✅
   전체 응답 길이: 67자

❓ 문제: A은 B보다 크고, B은 C보다 큽니다. A과 C 중 누가 더 큰가요?

🤖 모델 응답:
   <think>
A는 B보다 크다는 정보를 먼저 이해하겠습니다.
B는 C보다 큽니다라는 정보도 이해하겠습니다.
따라서 A와 C의 관계를 파악하기 위해 다음과 같이 진행하겠습니다.
</think>

<think>
A > B
B > C
</think>

두 조건을 합쳐보겠습니다:

A > B > C

따라서 A가 더 큽니다.

정답: A<|im_end|>

📊 패턴 분석:
   <think> 태그 사용: ✅
   <think> 내용 길이: 90자
   <think> 내용: A는 B보다 크다는 정보를 먼저 이해하겠습니다.
B는 C보다 큽니다라는 정보도 이해하겠습니다.
따라서 A와 C의 관계를 파악하기 위해 다음과 같이 진행하겠습니다.
   정답 형식: ✅
   전체 응답 길이: 196자

❓ 문제: 사과가 15개 있었는데 8개를 더 샀습니다. 총 사과는 몇 개인가요?

🤖 모델 응답:
   <think>
사과의 개수를 세는 것은 다음과 같이 진행할 수 있습니다:
1. 첫 번째 단계: 사과가 처음 가지고 있던 개수를 나타내기 위해 15를 사용합니다.
2. 두 번째 단계: 사과를 더한 개수를 나타내기 위해 8을 추가합니다.
3. 세 번째 단계: 두 개의 숫자를 더하여 총 사과의 개수를 구하는 것입니다.
</think>

정답: 23<|im_end|>

📊 패턴 분석:
   <t

In [18]:
# GRPO 학습의 영향 분석
print("=" * 60)
print("📌 GRPO 학습 영향 분석")
print("=" * 60)

print("""
📊 GRPO 학습으로 기대되는 변화:

1️⃣ 정답률 향상
   • 정답에 대한 보상(+1.0)으로 정확한 계산 능력 강화
   • 오답 패턴의 확률 감소

2️⃣ 형식 준수율 향상
   • 형식 보상(+0.5)으로 '정답: [숫자]' 패턴 학습
   • 일관된 출력 형식 유지

3️⃣ 풀이 과정 생성
   • 풀이 보상(+0.25)으로 단계적 설명 능력 향상
   • Chain-of-Thought 패턴 발현 가능

4️⃣ DeepSeek R1과의 차이
   • 규모: R1은 671B, 우리는 1.5B (약 450배 차이)
   • 데이터: R1은 수십만 문제, 우리는 100문제
   • 그룹 크기: R1은 G=64, 우리는 G=4
   • 학습 시간: R1은 수천 GPU-hour, 우리는 ~1시간
   
   → 하지만 GRPO의 핵심 원리는 동일합니다!
   → 더 많은 데이터와 더 큰 모델로 확장하면 성능이 크게 향상됩니다.
""")

📌 GRPO 학습 영향 분석

📊 GRPO 학습으로 기대되는 변화:

1️⃣ 정답률 향상
   • 정답에 대한 보상(+1.0)으로 정확한 계산 능력 강화
   • 오답 패턴의 확률 감소

2️⃣ 형식 준수율 향상
   • 형식 보상(+0.5)으로 '정답: [숫자]' 패턴 학습
   • 일관된 출력 형식 유지

3️⃣ 풀이 과정 생성
   • 풀이 보상(+0.25)으로 단계적 설명 능력 향상
   • Chain-of-Thought 패턴 발현 가능

4️⃣ DeepSeek R1과의 차이
   • 규모: R1은 671B, 우리는 1.5B (약 450배 차이)
   • 데이터: R1은 수십만 문제, 우리는 100문제
   • 그룹 크기: R1은 G=64, 우리는 G=4
   • 학습 시간: R1은 수천 GPU-hour, 우리는 ~1시간

   → 하지만 GRPO의 핵심 원리는 동일합니다!
   → 더 많은 데이터와 더 큰 모델로 확장하면 성능이 크게 향상됩니다.



In [19]:
# GPU 메모리 정리
print("🧹 GPU 메모리 최종 정리...")
print_gpu_memory("정리 전")

del grpo_trainer
del model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료")

🧹 GPU 메모리 최종 정리...
[정리 전] GPU: 2.9GB / 7.8GB
[정리 후] GPU: 0.5GB / 7.8GB
✅ 메모리 정리 완료


---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **DeepSeek R1 학습 과정**: Cold Start SFT → 추론 RL → Rejection Sampling + SFT → 전체 RL

2️⃣ **GRPO 알고리즘**: 그룹 내 상대 비교로 Value Model 없이 학습. 메모리 효율적.

3️⃣ **보상 함수 설계**: 정답 보상(+1.0) + 형식 보상(+0.5) + 풀이 보상(+0.25)

4️⃣ **RTX 4060 GRPO 설정**:
   - `num_generations=4` (그룹 크기)
   - `per_device_train_batch_size=1`
   - `gradient_accumulation_steps=8`
   - `max_completion_length=256`

5️⃣ **Chain-of-Thought 패턴**: GRPO 학습으로 모델이 단계적 추론 패턴을 학습

6️⃣ **"Aha Moment"**: 충분한 학습 시 모델이 자발적으로 자기 검증 능력을 획득

### Part 4 강화학습 전체 요약

| 세션 | 주제 | 핵심 |
|------|------|------|
| Session 25 | RL 개념 | PPO/DPO/GRPO 원리 이해 |
| Session 26 | Preference 데이터 | 선호도 데이터 수집/생성 |
| Session 26b | Rejection Sampling 개념 | Rejection Sampling 학습 |
| Session 27 | DPO 학습 | SFT → DPO 파이프라인 실습 |
| Session 28 | GRPO 학습 | DeepSeek R1 사례, 수학 추론 |

### 다음 세션 예고

- 📘 **Part 4**: 양자화, 배포 & 평가 세션
  - 양자화, API 서빙, Streamlit 웹앱, 평가 메트릭

In [20]:
# 최종 확인
print("=" * 60)
print("📌 최종 확인")
print("=" * 60)

# 저장된 체크포인트 확인
checkpoint_dir = OUTPUT_DIR / "grpo_checkpoint"
if checkpoint_dir.exists():
    files = list(checkpoint_dir.glob("*"))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
    print(f"  ✅ GRPO 체크포인트: {len(files)} 파일, {total_size:.1f}MB")
else:
    print(f"  ⚠️ GRPO 체크포인트: 디렉토리 없음")

print(f"\n📊 학습 결과 요약:")
print(f"   베이스라인 정확도: {baseline_acc:.1f}%")
print(f"   GRPO 후 정확도:   {after_acc:.1f}%")
print(f"   향상:             {after_acc - baseline_acc:+.1f}%")

print("\n" + "=" * 60)
print("✅ Session 28 완료!")
print("   DeepSeek R1 Case Study와 GRPO 학습을 성공적으로 실습했습니다.")
print("   Part 4 (강화학습) 전체 과정이 완료되었습니다!")

📌 최종 확인
  ✅ GRPO 체크포인트: 12 파일, 31.8MB

📊 학습 결과 요약:
   베이스라인 정확도: 80.0%
   GRPO 후 정확도:   100.0%
   향상:             +20.0%

✅ Session 28 완료!
   DeepSeek R1 Case Study와 GRPO 학습을 성공적으로 실습했습니다.
   Part 4 (강화학습) 전체 과정이 완료되었습니다!
